## Librerias

In [39]:
import pandas as pd 

In [40]:
data_path = "/Users/ekaterinasorokopudova/Desktop/simulador/ProjecteData/Equip_21/Data/Bronze_BANK_marketing_020226.csv"

In [41]:
df = pd.read_csv(
    data_path,
    sep=",",
    encoding="utf-8"
)

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         11162 non-null  int64  
 1   age        11152 non-null  float64
 2   job        11162 non-null  object 
 3   marital    11157 non-null  object 
 4   education  11155 non-null  object 
 5   default    11162 non-null  object 
 6   balance    11162 non-null  int64  
 7   housing    11162 non-null  object 
 8   loan       11162 non-null  object 
 9   contact    11162 non-null  object 
 10  day        11162 non-null  int64  
 11  month      11162 non-null  object 
 12  duration   11162 non-null  int64  
 13  campaign   11162 non-null  int64  
 14  pdays      11162 non-null  int64  
 15  previous   11162 non-null  int64  
 16  poutcome   11162 non-null  object 
 17  deposit    11162 non-null  object 
dtypes: float64(1), int64(7), object(10)
memory usage: 1.5+ MB


In [43]:
df.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,2,56.0,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,3,41.0,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


## 1. Tratamiento de valores NULL 

In [44]:
columns_with_null = df.columns[df.isnull().any()].tolist()
columns_with_null

['age', 'marital', 'education']

In [45]:
df["age"] = df["age"].fillna(df["age"].median())
df["age"] = df["age"].astype(int)  

In [46]:
df['marital'].value_counts()


marital
married     6349
single      3517
divorced    1291
Name: count, dtype: int64

In [47]:
df[df["marital"].isna()]


,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
1088,1089,53,management,NaN,secondary,no,1004,no,yes,telephone,22,dec,119,1,-1,0,unknown,yes
3719,3720,68,retired,NaN,secondary,no,4189,no,no,telephone,14,jul,897,2,-1,0,unknown,yes
4854,4855,34,management,NaN,tertiary,no,5,no,no,cellular,18,aug,370,2,-1,0,unknown,yes
6701,6702,60,admin.,NaN,primary,no,-444,no,yes,cellular,16,jul,227,1,-1,0,unknown,no
9272,9273,30,blue-collar,NaN,primary,no,35,yes,no,cellular,11,jul,366,2,-1,0,unknown,no


In [48]:
df["education"].value_counts()

education
secondary    5474
tertiary     3685
primary      1500
unknown       496
Name: count, dtype: int64

In [49]:
df[df["education"].isna()]


,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
1300,1301,37,technician,married,NaN,no,549,no,no,cellular,2,mar,239,1,-1,0,unknown,yes
2943,2944,24,student,single,NaN,no,382,no,no,cellular,13,nov,256,2,92,3,failure,yes
4737,4738,37,management,single,NaN,no,102,yes,no,cellular,6,may,445,1,258,2,failure,yes
6819,6820,32,services,married,NaN,no,-344,no,yes,cellular,4,feb,44,1,-1,0,unknown,no
7121,7122,57,entrepreneur,married,NaN,no,657,no,no,unknown,12,jun,344,1,-1,0,unknown,no
8592,8593,55,management,single,NaN,no,797,no,no,cellular,29,jul,24,2,-1,0,unknown,no
9406,9407,32,technician,single,NaN,no,696,no,yes,cellular,13,aug,101,4,105,4,failure,no


In [50]:
df = df.dropna(subset=["marital", "education"])

## 2. Tratamiento de valores Unknown 

In [51]:
unknown_df = (
    (df == "unknown")
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "unknown_count"})
)

unknown_df = unknown_df[unknown_df["unknown_count"] > 0]
unknown_df

,column,unknown_count
2,job,70
4,education,496
9,contact,2345
16,poutcome,8317


- job - "not_provided"
- education - "not_provided"
- contact - "undefined_contact_type"
- poutcome - "new_marketing_client"


job (70 строк): оставить как not_provided (или unknown) — влияния почти нет.
education (496 строк): тоже можно оставить, если кластеры нормально интерпретируются.
Если потом увидишь кластер “not_provided-heavy” и он мешает — тогда включать Вариант B.

job (70 líneas): dejar como not_provided (o desconocido) – casi sin impacto.
education (496 líneas): también se puede dejar siempre que los clústeres se interpreten correctamente.
Si posteriormente ve el clúster "not_provided-heavy" y está interfiriendo, utilice la opción B.

In [52]:
df = df.replace({
    "job": {"unknown": "not_provided"},
    "education": {"unknown": "not_provided"},
    "contact": {"unknown": "undefined_contact_type"},
    "poutcome": {"unknown": "new_marketing_client"}
})

In [53]:
(df[["job", "education", "contact", "poutcome"]] == "unknown").sum()

job          0
education    0
contact      0
poutcome     0
dtype: int64

#### Opcion B 

In [ ]:
import numpy as np
import pandas as pd

In [ ]:

df["age_bin"] = pd.cut(df["age"], bins=[0, 25, 35, 45, 55, 65, 120], right=False)

In [ ]:

def fill_unknown_by_group_mode(df, target_col, group_cols, unknown="unknown"):
    mask = df[target_col].eq(unknown)

    mode_in_group = (
        df.loc[~df[target_col].eq(unknown)]
        .groupby(group_cols)[target_col]
        .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    )

    keys = [df.loc[mask, c] for c in group_cols]
    df.loc[mask, target_col] = mode_in_group.reindex(pd.MultiIndex.from_arrays(keys)).to_numpy()

    global_mode = df.loc[~df[target_col].eq(unknown), target_col].mode().iat[0]
    df[target_col] = df[target_col].fillna(global_mode)

    return df

In [ ]:
df = fill_unknown_by_group_mode(df, "job", ["education", "marital", "age_bin"])
df = fill_unknown_by_group_mode(df, "education", ["job", "marital", "age_bin"])

df = df.drop(columns=["age_bin"])

## 3. Validación y corrección de valores atípicos

In [54]:
df[["age", "day", "campaign", "previous"]].describe().round(2)

,age,day,campaign,previous
count,11150.00,11150.00,11150.00,11150.00
mean,41.23,15.66,2.51,0.83
std,11.91,8.42,2.72,2.29
min,18.00,1.00,1.00,0.00
25%,32.00,8.00,1.00,0.00
50%,39.00,15.00,2.00,0.00
75%,49.00,22.00,3.00,1.00
max,95.00,31.00,63.00,58.00


1. ¿Existen límites naturales?
Existen límites naturales razonables para cada indicador: La edad no debe ser mayor de 95 años, el día no debe ser menor de 1 ni mayor de 31, el número de llamadas siempre es positivo, al igual que el número de contactos previos con el cliente.

2. ¿Es un valor medio o un código?
Todos los datos mencionados son mediciones, no códigos.

3. ¿Es real el extremo?
Todos los valores máximos son reales.

In [55]:
df["pdays"].value_counts().head()

pdays
-1      8315
 92      105
 182      89
 91       84
 181      81
Name: count, dtype: int64

La variable no muestra mediciones, sino un código que responde a la pregunta:
¿Hubo una llamada?
1. Sí: ¿Cuántos días han pasado? >90 indica el número exacto de días; >180 indica el número exacto de días.
2. No: Ingrese -1.

In [56]:
df["balance"].describe().round(2)

count    11150.00
mean      1529.50
std       3226.79
min      -6847.00
25%        123.00
50%        550.00
75%       1709.75
max      81204.00
Name: balance, dtype: float64

1. ¿Existen límites naturales?
No. El importe en la cuenta del cliente puede ser de cualquier valor.

2. ¿Es un valor medio o un código?
Es una medida.

3. ¿Puede el extremo ser real?
Sí. Cualquier valor.

# 4. Estandarización de formatos

In [57]:
df["age"] = df["age"].astype(int)

In [58]:
df["balance"] = df["balance"].astype(float)

In [59]:
df.dtypes

id             int64
age            int64
job           object
marital       object
education     object
default       object
balance      float64
housing       object
loan          object
contact       object
day            int64
month         object
duration       int64
campaign       int64
pdays          int64
previous       int64
poutcome      object
deposit       object
dtype: object

# 5. Duplicados

- Duplicados completos → eliminar
- Duplicados por todos los atributos → eliminar
- Duplicados por ID → no siempre duplicados

#### Duplicados completos de líneas (coincidencia del 100%)

In [60]:
df.duplicated().sum()

np.int64(0)

In [61]:
df[df.duplicated(keep=False)]

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit


In [62]:
df = df.drop_duplicates()

#### Duplicados por subconjunto de columnas (duplicado parcial)

In [63]:
key_cols = [
    "age", "job", "marital", "education",
    "balance", "housing", "loan",
    "contact", "day", "month",
    "duration", "campaign", "pdays",
    "previous", "poutcome", "deposit"
]

df.duplicated(subset=key_cols).sum()

np.int64(0)

In [64]:
df[df.duplicated(subset=key_cols, keep=False)]

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit


In [ ]:
df = df.drop_duplicates(subset=key_cols, keep="first")

#### Duplicados por cliente (punto importante!)

In [65]:
df["id"].duplicated().sum()

np.int64(0)

In [66]:
df[df["id"].duplicated(keep=False)].sort_values("id")

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit


# Validacióm final 

In [67]:
df_raw = pd.read_csv(data_path) 

before_rows = len(df_raw)

after_rows = len(df)

removed_rows = before_rows - after_rows

print("=== DATA CLEANING VALIDATION ===")
print(f"Rows before: {before_rows}")
print(f"Rows after : {after_rows}")
print(f"Removed    : {removed_rows} ({removed_rows / before_rows:.2%})")
print()

=== DATA CLEANING VALIDATION ===
Rows before: 11162
Rows after : 11150
Removed    : 12 (0.11%)



In [ ]:
# NULL summary 
null_counts = df.isna().sum()
null_summary = (
    null_counts[null_counts > 0]
    .sort_values(ascending=False)
    .rename("null_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

print("=== NULLS (only columns with >0) ===")
display(null_summary)
print()

=== NULLS (only columns with >0) ===


,column,null_count


In [69]:
# UNKNOWN summary
unknown_counts = (df == "unknown").sum(numeric_only=False)
unknown_summary = (
    unknown_counts[unknown_counts > 0]
    .sort_values(ascending=False)
    .rename("unknown_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

print("=== 'unknown' (only columns with >0) ===")
display(unknown_summary)
print()


=== 'unknown' (only columns with >0) ===


,column,unknown_count


In [ ]:
# Types
types_summary = (
    df.dtypes.astype(str)
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)

print("=== DTYPES ===")
display(types_summary)
print()


=== DTYPES ===


,column,dtype
0,id,int64
1,age,int64
2,job,object
3,marital,object
4,education,object
5,default,object
6,balance,float64
7,housing,object
8,loan,object
9,contact,object


# Exportar un conjunto de datos terminado

In [71]:
df.to_csv(
    "/Users/ekaterinasorokopudova/Desktop/simulador/ProjecteData/Equip_21/Data/Silver_BANK_marketing_020226.csv",
    index=False,
    encoding="utf-8"
)